## MapleFreight Delivery Delay Prediction

## 1. Business Context

MapleFreight Logistics moves about 8,000 shipments a week across Ontario, Quebec and the Prairies. Its contracts carry on-time guarantees, so every late delivery costs service credits and re-delivery charges.

On-time performance has dropped from 94% to 88% this year, and we only learn a shipment is late when the customer calls.

## 2. Problem Statement

You are a Data Scientist at MapleFreight. Delays are currently found only after the delivery window has passed, which means paid-out service credits, expedited re-deliveries and complaints that reach the account manager before dispatch knows anything is wrong.

Build an end-to-end system that predicts which in-transit shipments will arrive late and alerts dispatch in Slack while there is still time to re-route or warn the customer.

## 3. Objectives
1. Analyze shipment, route and carrier data to identify the main drivers of late delivery.
2. Build a predictive model that classifies whether a shipment will be delivered late.
3. Integrate Slack alerts so dispatch is notified about high-risk shipments automatically.
4. Provide actionable recommendations that improve on-time performance.

## 4. Data Understanding

In [55]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [56]:
df = pd.read_csv("maplefreight_delivery_delay_dataset.csv")
df.head(10)

,shipment_id,origin_city,destination_city,carrier,service_level,goods_type,customer_priority_tier,shipment_month,distance_km,weight_kg,...,pickup_delay_minutes,driver_experience_years,vehicle_age_years,weather_condition,traffic_index,route_congestion_score,prior_late_deliveries_30d,fuel_cost_cad,actual_transit_hours,delivered_late
0,SHP-103848,Mississauga ON,Edmonton AB,NorthStar Carriers,Standard,General Freight,Bronze,1,376.0,257.1,...,55.0,5.1,6.1,Fog,37.2,0.291,1,146.39,7.55,0
1,SHP-102827,Montreal QC,Winnipeg MB,PrairieLine,Standard,General Freight,Silver,12,159.8,734.1,...,69.0,5.3,4.2,Snow,77.4,0.236,0,155.07,1.17,1
2,SHP-105254,Toronto ON,Hamilton ON,QuebecExpress,Standard,Fragile,Silver,9,423.0,680.8,...,-13.0,1.4,2.8,Rain,36.5,0.344,0,163.44,6.47,0
3,SHP-102719,Montreal QC,Hamilton ON,MapleFreight Fleet,Standard,General Freight,Silver,1,494.1,1292.2,...,-20.0,4.0,14.8,Fog,51.2,0.491,2,NaN,8.03,0
4,SHP-105198,Mississauga ON,Sudbury ON,MapleFreight Fleet,Standard,Refrigerated,Bronze,9,843.5,3327.1,...,28.0,2.9,1.6,Clear,82.7,0.423,1,263.22,13.49,0
5,SHP-101857,Montreal QC,Winnipeg MB,Contract Owner-Op,Express,General Freight,Gold,10,572.7,1310.8,...,53.0,3.7,5.3,Storm,72.9,0.467,4,149.19,10.00,1
6,SHP-102142,Toronto ON,Toronto ON,QuebecExpress,Economy,General Freight,Silver,6,303.3,1259.5,...,-5.0,4.5,3.2,Clear,65.2,0.514,0,133.83,5.32,0
7,SHP-103123,Mississauga ON,Winnipeg MB,NorthStar Carriers,Standard,General Freight,Gold,1,776.6,1043.6,...,9.0,4.9,4.2,NaN,58.9,0.586,0,180.57,11.60,0
8,SHP-100691,Mississauga ON,Quebec City QC,PrairieLine,Express,General Freight,Silver,9,755.4,889.2,...,25.0,2.3,4.7,Clear,54.4,0.330,1,150.68,10.77,0
9,SHP-101694,London ON,Regina SK,MapleFreight Fleet,Express,Fragile,Silver,9,172.7,1322.8,...,38.0,10.2,2.4,Clear,63.6,0.778,1,124.74,3.26,0


In [57]:
df.shape

(6035, 23)

In [58]:
df.describe()

,shipment_month,distance_km,weight_kg,num_stops,is_cross_border,scheduled_transit_hours,pickup_delay_minutes,driver_experience_years,vehicle_age_years,traffic_index,route_congestion_score,prior_late_deliveries_30d,fuel_cost_cad,actual_transit_hours,delivered_late
count,6035.000000,6035.000000,6.035000e+03,6035.000000,6035.000000,6035.000000,6035.000000,5825.000000,6035.000000,5914.000000,6035.000000,6035.000000,5795.000000,6035.000000,6035.000000
mean,6.543662,632.192742,6.928251e+03,2.477051,0.113007,10.500388,20.367688,6.497648,5.489461,55.146365,0.449778,1.305551,180.696552,11.186038,0.205800
std,3.448260,319.246133,8.375515e+04,1.707457,0.316628,4.758576,30.618617,4.659277,3.433676,17.592978,0.194500,1.158943,58.759928,5.957114,0.404318
min,1.000000,80.500000,3.000000e+01,0.000000,0.000000,1.000000,-20.000000,0.200000,0.100000,5.000000,0.000000,0.000000,30.000000,0.880000,0.000000
25%,4.000000,398.600000,4.960000e+02,1.000000,0.000000,7.060000,-6.000000,3.200000,3.000000,43.300000,0.318000,0.000000,140.595000,7.080000,0.000000
50%,7.000000,571.100000,8.271000e+02,2.000000,0.000000,9.850000,18.000000,5.400000,4.800000,55.000000,0.446000,1.000000,181.180000,10.050000,0.000000
75%,10.000000,804.850000,1.286300e+03,4.000000,0.000000,13.175000,41.000000,8.600000,7.200000,67.200000,0.583000,2.000000,220.950000,14.010000,0.000000
max,12.000000,2556.700000,2.999300e+06,5.000000,1.000000,37.400000,146.000000,35.000000,20.000000,100.000000,1.000000,8.000000,392.010000,55.380000,1.000000


In [59]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6035 entries, 0 to 6034
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   shipment_id                6035 non-null   str    
 1   origin_city                6035 non-null   str    
 2   destination_city           6035 non-null   str    
 3   carrier                    6035 non-null   str    
 4   service_level              6035 non-null   str    
 5   goods_type                 6035 non-null   str    
 6   customer_priority_tier     6035 non-null   str    
 7   shipment_month             6035 non-null   int64  
 8   distance_km                6035 non-null   float64
 9   weight_kg                  6035 non-null   float64
 10  num_stops                  6035 non-null   int64  
 11  is_cross_border            6035 non-null   int64  
 12  scheduled_transit_hours    6035 non-null   float64
 13  pickup_delay_minutes       6035 non-null   float64
 14  dri

In [60]:
df['carrier'].value_counts()

carrier
MapleFreight Fleet    2391
NorthStar Carriers    1164
PrairieLine            932
QuebecExpress          924
Contract Owner-Op      624
Name: count, dtype: int64

In [61]:
df['service_level'].value_counts()

service_level
Standard    3219
Express     1442
Economy     1254
standard      74
express       27
economy       19
Name: count, dtype: int64

Potential issue: Inconsistent category labels may cause the model to treat equivalent categories as different groups, which can negatively affect its ability to generalize to new data.

In [62]:
#number of rows duplicated in the dataset
df.duplicated().sum()   

np.int64(32)

In [63]:
#Shows the information for the 32 duplicates
df[df.duplicated(keep='first')]

,shipment_id,origin_city,destination_city,carrier,service_level,goods_type,customer_priority_tier,shipment_month,distance_km,weight_kg,...,pickup_delay_minutes,driver_experience_years,vehicle_age_years,weather_condition,traffic_index,route_congestion_score,prior_late_deliveries_30d,fuel_cost_cad,actual_transit_hours,delivered_late
932,SHP-100457,Montreal QC,Toronto ON,PrairieLine,Economy,General Freight,Silver,8,532.1,1113.3,...,26.0,20.2,4.6,Clear,66.3,0.691,0,115.19,8.49,0
1443,SHP-103447,Toronto ON,Calgary AB,MapleFreight Fleet,Standard,Bulk,Bronze,11,430.5,1018.4,...,5.0,4.9,9.3,Snow,24.2,0.468,3,106.86,10.49,0
2453,SHP-104541,London ON,Ottawa ON,NorthStar Carriers,Standard,General Freight,Gold,11,608.8,637.0,...,-1.0,3.1,3.8,Clear,62.7,0.441,1,193.43,8.97,0
2599,SHP-100037,Mississauga ON,Calgary AB,MapleFreight Fleet,Economy,General Freight,Silver,9,605.8,864.1,...,37.0,3.3,8.4,Clear,49.8,0.315,2,262.41,10.05,0
2920,SHP-101174,London ON,Edmonton AB,MapleFreight Fleet,Standard,Refrigerated,Bronze,4,348.2,883.6,...,6.0,12.8,3.1,Clear,65.3,0.420,1,208.64,4.82,0
2956,SHP-103516,Toronto ON,Sudbury ON,PrairieLine,Express,General Freight,Gold,10,215.3,603.7,...,9.0,6.1,5.0,Rain,79.3,0.600,1,211.34,7.80,1
3135,SHP-104402,Winnipeg MB,Sudbury ON,NorthStar Carriers,Express,Refrigerated,Bronze,7,855.0,1098.4,...,-20.0,3.1,4.4,Storm,NaN,0.713,1,152.77,14.64,0
3184,SHP-101699,Toronto ON,Quebec City QC,QuebecExpress,Express,General Freight,Bronze,9,523.3,775.9,...,30.0,10.6,5.5,Clear,75.7,0.636,1,239.41,9.88,0
3188,SHP-105617,Toronto ON,Quebec City QC,MapleFreight Fleet,Standard,General Freight,Silver,12,465.2,928.4,...,4.0,3.8,3.8,Fog,69.2,0.612,1,197.80,9.61,0
3583,SHP-100018,Winnipeg MB,Sudbury ON,QuebecExpress,Economy,Fragile,Bronze,7,426.1,556.3,...,20.0,1.3,4.4,Snow,92.9,0.386,0,320.13,8.81,0


In [81]:
#verify f in any other column there is an inconsistency
for col in df.select_dtypes(include='object').columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- shipment_id ---
shipment_id
SHP-102224    2
SHP-100457    2
SHP-104541    2
SHP-103447    2
SHP-105761    2
             ..
SHP-101294    1
SHP-104024    1
SHP-105201    1
SHP-103776    1
SHP-101946    1
Name: count, Length: 6000, dtype: int64

--- origin_city ---
origin_city
Toronto ON        1593
Mississauga ON    1019
Montreal QC       1017
Calgary AB         762
Ottawa ON          730
London ON          460
Winnipeg MB        454
Name: count, dtype: int64

--- destination_city ---
destination_city
Sudbury ON        641
Toronto ON        638
Regina SK         634
Winnipeg MB       614
Hamilton ON       604
Quebec City QC    598
Edmonton AB       592
Montreal QC       587
Calgary AB        578
Ottawa ON         549
Name: count, dtype: int64

--- carrier ---
carrier
MapleFreight Fleet    2391
NorthStar Carriers    1164
PrairieLine            932
QuebecExpress          924
Contract Owner-Op      624
Name: count, dtype: int64

--- service_level ---
service_level
Standard    3293
Ex

C:\Users\LILIP\AppData\Local\Temp\ipykernel_21352\3243900158.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [ ]:
#The verification by column let me know that for the shipment_id there is 6000 unique values but we have 6035 raws, so we need to investigate if they are duplicate or something else
df['shipment_id'].duplicated().sum()

np.int64(35)

In [65]:
#Verify if the duplicates contains the same information ordered by shipment_id, since we have 35
df[df['shipment_id'].duplicated(keep=False)].sort_values('shipment_id')

,shipment_id,origin_city,destination_city,carrier,service_level,goods_type,customer_priority_tier,shipment_month,distance_km,weight_kg,...,pickup_delay_minutes,driver_experience_years,vehicle_age_years,weather_condition,traffic_index,route_congestion_score,prior_late_deliveries_30d,fuel_cost_cad,actual_transit_hours,delivered_late
1921,SHP-100018,Winnipeg MB,Sudbury ON,QuebecExpress,Economy,Fragile,Bronze,7,426.1,556.3,...,20.0,1.3,4.4,Snow,92.9,0.386,0,320.13,8.81,0
3583,SHP-100018,Winnipeg MB,Sudbury ON,QuebecExpress,Economy,Fragile,Bronze,7,426.1,556.3,...,20.0,1.3,4.4,Snow,92.9,0.386,0,320.13,8.81,0
2599,SHP-100037,Mississauga ON,Calgary AB,MapleFreight Fleet,Economy,General Freight,Silver,9,605.8,864.1,...,37.0,3.3,8.4,Clear,49.8,0.315,2,262.41,10.05,0
867,SHP-100037,Mississauga ON,Calgary AB,MapleFreight Fleet,Economy,General Freight,Silver,9,605.8,864.1,...,37.0,3.3,8.4,Clear,49.8,0.315,2,262.41,10.05,0
4115,SHP-100063,Ottawa ON,Quebec City QC,MapleFreight Fleet,Standard,Hazardous,Bronze,6,1255.7,844.8,...,48.0,4.3,11.4,Clear,41.9,0.338,1,211.74,20.18,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2198,SHP-105433,Toronto ON,Ottawa ON,QuebecExpress,Express,General Freight,Silver,12,935.7,999.0,...,7.0,9.8,1.6,Rain,53.4,0.369,2,117.48,15.33,0
702,SHP-105617,Toronto ON,Quebec City QC,MapleFreight Fleet,Standard,General Freight,Silver,12,465.2,928.4,...,4.0,3.8,3.8,Fog,69.2,0.612,1,197.80,9.61,0
3188,SHP-105617,Toronto ON,Quebec City QC,MapleFreight Fleet,Standard,General Freight,Silver,12,465.2,928.4,...,4.0,3.8,3.8,Fog,69.2,0.612,1,197.80,9.61,0
307,SHP-105761,Mississauga ON,Montreal QC,NorthStar Carriers,Standard,Refrigerated,Bronze,6,303.7,68.9,...,31.0,8.0,2.5,Clear,43.0,0.723,1,235.19,11.98,1


Finding: The dataset contains 32 additional duplicate rows, but also we found that there are 35 duplicates for shipment_id that probably include those 32 duplicates, before removing them will be better to investigate if they are exactly the same since keeping them may give duplicated observations more influence during model training and may lead to data leakage between the training and test sets.

In [67]:
# Count missing values in a specific column
df.isna().sum()

shipment_id                    0
origin_city                    0
destination_city               0
carrier                        0
service_level                  0
goods_type                     0
customer_priority_tier         0
shipment_month                 0
distance_km                    0
weight_kg                      0
num_stops                      0
is_cross_border                0
scheduled_transit_hours        0
pickup_delay_minutes           0
driver_experience_years      210
vehicle_age_years              0
weather_condition            150
traffic_index                121
route_congestion_score         0
prior_late_deliveries_30d      0
fuel_cost_cad                240
actual_transit_hours           0
delivered_late                 0
dtype: int64

In [68]:
#Ho much does missing values represent vs the total data
round((df.isna().sum())/df.__len__()*100,1)

shipment_id                  0.0
origin_city                  0.0
destination_city             0.0
carrier                      0.0
service_level                0.0
goods_type                   0.0
customer_priority_tier       0.0
shipment_month               0.0
distance_km                  0.0
weight_kg                    0.0
num_stops                    0.0
is_cross_border              0.0
scheduled_transit_hours      0.0
pickup_delay_minutes         0.0
driver_experience_years      3.5
vehicle_age_years            0.0
weather_condition            2.5
traffic_index                2.0
route_congestion_score       0.0
prior_late_deliveries_30d    0.0
fuel_cost_cad                4.0
actual_transit_hours         0.0
delivered_late               0.0
dtype: float64

In [69]:
for col in df.select_dtypes(include='float').columns:
    ng= df[col] < 0
    print(f"{col}: {ng.sum()}")

distance_km: 0
weight_kg: 0
scheduled_transit_hours: 0
pickup_delay_minutes: 1794
driver_experience_years: 0
vehicle_age_years: 0
traffic_index: 0
route_congestion_score: 0
fuel_cost_cad: 0
actual_transit_hours: 0


Finding: pickup_delay_minutes contains 1,794 negative values, while the other numerical variables contain no negative values. These values should be investigated before being treated as errors because negative pickup delays may represent pickups occurring earlier than scheduled.

In [70]:
df['weight_kg'].describe()

count    6.035000e+03
mean     6.928251e+03
std      8.375515e+04
min      3.000000e+01
25%      4.960000e+02
50%      8.271000e+02
75%      1.286300e+03
max      2.999300e+06
Name: weight_kg, dtype: float64

In [71]:
Q1 = df['weight_kg'].quantile(0.25)
Q3 = df['weight_kg'].quantile(0.75)
IQR = Q3 - Q1
print(IQR)

790.3000000000002


In [72]:
upper_limit = Q3 + 1.5 * IQR
print(upper_limit)

2471.7500000000005


In [73]:
print((df['weight_kg'] > upper_limit).sum())

189


In [74]:
df[df['weight_kg'] > upper_limit][['shipment_id', 'weight_kg', 'goods_type', 'delivered_late']]

,shipment_id,weight_kg,goods_type,delivered_late
4,SHP-105198,3327.1,Refrigerated,0
10,SHP-102224,3993.1,Bulk,0
12,SHP-101568,2659.2,General Freight,0
26,SHP-104494,2552.4,General Freight,0
87,SHP-104430,2823.2,General Freight,1
...,...,...,...,...
5964,SHP-100930,3401.2,General Freight,0
5967,SHP-104257,2535.5,Bulk,0
5995,SHP-105366,2557.6,Hazardous,0
6003,SHP-103096,2755.6,Fragile,1


In [75]:
df['weight_kg'].nlargest(10)

2323    2999300.0
846     1593600.0
1525    1554700.0
3284    1408300.0
4761    1389500.0
696     1332200.0
4028    1246100.0
4876    1238200.0
4024    1227100.0
1030    1209400.0
Name: weight_kg, dtype: float64

In [76]:
# simplest fix: keep the 10 largest weight values
top10 = df.loc[df['weight_kg'].nlargest(10).index, ['weight_kg', 'goods_type']]
print(top10)

      weight_kg       goods_type
2323  2999300.0        Hazardous
846   1593600.0  General Freight
1525  1554700.0  General Freight
3284  1408300.0  General Freight
4761  1389500.0  General Freight
696   1332200.0  General Freight
4028  1246100.0  General Freight
4876  1238200.0          Fragile
4024  1227100.0     Refrigerated
1030  1209400.0  General Freight


Finding: weight_kg contains several extremely large values that are far above the typical shipment weights. These values occur across multiple goods types, so they should be investigated as potential data-quality or unit issues rather than automatically removed as outliers.

## 5. Exploratory data analysis (EDA)

 late vs on-time
 carrier
 service level
 weather
 traffic
 congestion
 pickup delay
 previous late deliveries
 route/distance

In [77]:
df['delivered_late'].value_counts(normalize=True)*100

delivered_late
0    79.42005
1    20.57995
Name: proportion, dtype: float64

The target variable delivered_late shows that approximately 21% of shipments are delivered late, while the remaining 79% are delivered on time. This means that the dataset is unbalanced and this could affect the model

In [78]:
df.groupby('service_level')['delivered_late'].mean().sort_values(ascending=False)

service_level
Economy     0.265550
economy     0.263158
express     0.222222
standard    0.216216
Standard    0.206897
Express     0.149792
Name: delivered_late, dtype: float64

In [79]:
df.groupby('carrier')['delivered_late'].mean().sort_values(ascending=False)

carrier
Contract Owner-Op     0.358974
PrairieLine           0.265021
QuebecExpress         0.224026
NorthStar Carriers    0.219931
MapleFreight Fleet    0.128816
Name: delivered_late, dtype: float64

## 6. Data preprocessing

In [80]:
service_level_map = {
    'Standard' : 'Standard',
    'standard' : 'Standard',
    'Economy' : 'Economy',
    'economy' : 'Economy',
    'Express' : 'Express',
    'express' : 'Express'
}

df['service_level'] = df['service_level'].map(service_level_map)
df['service_level'].value_counts()

service_level
Standard    3293
Express     1469
Economy     1273
Name: count, dtype: int64

## 7. Feature engineering

## 8. Data preparation

## 9. Data modelling

## 10. Model evaluation

## 11. Model explainability

## 12. Slack alert integration

## 13. Recommendations